# AtliQ Phase 2 (LEARNER STARTER) — Kafka → Delta with Structured Streaming
Complete the TODOs to build Bronze → Silver → Gold as **streams**.

**Free Edition (serverless) rules:** checkpoints go in a Unity Catalog
**Volume** (no DBFS), streams write to **managed tables**, and every stream
needs its **own** checkpoint folder.

In [0]:
KAFKA_BOOTSTRAP = "pkc-7prvp.centralindia.azure.confluent.cloud:9092"
KAFKA_TOPIC = "atliq.orders.events"
KAFKA_API_KEY = dbutils.secrets.get(scope="kafka-secrets", key="kafka-api-key")
KAFKA_API_SECRET = dbutils.secrets.get(scope="kafka-secrets", key="kafka-api-secret")

CATALOG, SCHEMA = "atliq", "streaming"
CKPT = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

# One-time setup (given):
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints")

DataFrame[]

## TASK 1 — Bronze: raw events off Kafka, no parsing
Read the topic with `spark.readStream.format("kafka")` and append the raw
records to `atliq.streaming.bronze_order_events`.

Hints:
- Options you need: `kafka.bootstrap.servers`, `subscribe`,
  `startingOffsets = earliest`, `kafka.security.protocol = SASL_SSL`,
  `kafka.sasl.mechanism = PLAIN`, and `kafka.sasl.jaas.config`
  (on Databricks the login module class is
  `kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule`).
- Kafka gives you binary key/value — CAST both to STRING.
- Keep topic, partition, offset, timestamp columns too. Bronze keeps everything.
- writeStream: outputMode "append", checkpointLocation f"{CKPT}/bronze",
  .toTable(...)

In [0]:
from pyspark.sql.types import StructType, StringType, IntegerType, DecimalType, LongType, TimestampType
from pyspark.sql import functions as F

jaas_config = (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="{KAFKA_API_KEY}" password="{KAFKA_API_SECRET}";'
)

raw_kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

bronze_df = (raw_kafka_df
                .selectExpr("CAST(key AS STRING) as kafka_key", 
                            "CAST(value AS STRING) as json_value",
                            "offset",
                            "topic",
                            "partition",
                            "timestamp")
            )

query = (
            (
                bronze_df.writeStream
                    .format('delta')
                    .outputMode('append')
                    .trigger(availableNow=True)
                    .option("checkpointLocation", f"{CKPT}/bronze")
                    .toTable(f"{CATALOG}.{SCHEMA}.bronze_order_events")
            )
        )

query.awaitTermination()

bronze_order_events_count = spark.table(f"{CATALOG}.{SCHEMA}.bronze_order_events").count()

print(f"Bronze order events count: {bronze_order_events_count}")

display(
    spark.table(f"{CATALOG}.{SCHEMA}.bronze_order_events")
    .limit(10)
)


Bronze order events count: 5321


kafka_key,json_value,offset,topic,partition,timestamp
100005,"{""event_id"": ""e1732958-7a1a-4224-874a-75d91abd8522"", ""event_type"": ""order_placed"", ""event_ts"": ""2026-08-27T01:04:35.629Z"", ""order_id"": 100005, ""customer_id"": 28, ""city"": ""Jaipur"", ""product_id"": 14, ""quantity"": 1, ""order_amount"": 1599, ""payment_method"": null}",828,atliq.orders.events,0,2026-08-27T01:04:35.629Z
100005,"{""event_id"": ""5179be37-3ff2-4448-9c2a-c2b5bce04e0b"", ""event_type"": ""payment_received"", ""event_ts"": ""2026-08-27T01:04:35.629Z"", ""order_id"": 100005, ""customer_id"": 28, ""city"": ""Jaipur"", ""product_id"": 14, ""quantity"": 1, ""order_amount"": 1599, ""payment_method"": ""Net Banking""}",829,atliq.orders.events,0,2026-08-27T01:04:35.629Z
100006,"{""event_id"": ""60451c03-cc20-4bae-9527-4d76f73769c4"", ""event_type"": ""order_placed"", ""event_ts"": ""2026-08-27T01:04:36.640Z"", ""order_id"": 100006, ""customer_id"": 38, ""city"": ""Chennai"", ""product_id"": 4, ""quantity"": 2, ""order_amount"": 2998, ""payment_method"": null}",830,atliq.orders.events,0,2026-08-27T01:04:36.640Z
100006,"{""event_id"": ""f4c90eee-a7bc-4a5b-945d-b29a7b57a8d4"", ""event_type"": ""payment_received"", ""event_ts"": ""2026-08-27T01:04:36.640Z"", ""order_id"": 100006, ""customer_id"": 38, ""city"": ""Chennai"", ""product_id"": 4, ""quantity"": 2, ""order_amount"": 2998, ""payment_method"": ""Wallet""}",831,atliq.orders.events,0,2026-08-27T01:04:36.640Z
100007,"{""event_id"": ""45b4315d-e24b-4ff9-9002-3d8c5fe123b4"", ""event_type"": ""order_placed"", ""event_ts"": ""2026-08-27T01:04:37.144Z"", ""order_id"": 100007, ""customer_id"": 14, ""city"": ""Hyderabad"", ""product_id"": 6, ""quantity"": 1, ""order_amount"": 899, ""payment_method"": null}",832,atliq.orders.events,0,2026-08-27T01:04:37.144Z
100011,"{""event_id"": ""b8212191-e724-44b2-978a-11137b05e04f"", ""event_type"": ""order_placed"", ""event_ts"": ""2026-08-27T01:04:39.166Z"", ""order_id"": 100011, ""customer_id"": 31, ""city"": ""Surat"", ""product_id"": 19, ""quantity"": 1, ""order_amount"": 799, ""payment_method"": null}",833,atliq.orders.events,0,2026-08-27T01:04:39.167Z
100011,"{""event_id"": ""1796ada0-9a7b-47e4-b2ae-5004413b4cbf"", ""event_type"": ""payment_received"", ""event_ts"": ""2026-08-27T01:04:39.166Z"", ""order_id"": 100011, ""customer_id"": 31, ""city"": ""Surat"", ""product_id"": 19, ""quantity"": 1, ""order_amount"": 799, ""payment_method"": ""Wallet""}",834,atliq.orders.events,0,2026-08-27T01:04:39.167Z
100006,"{""event_id"": ""62f2c951-3e8d-41a4-b4d9-dbe942669445"", ""event_type"": ""order_shipped"", ""event_ts"": ""2026-08-27T01:04:41.186Z"", ""order_id"": 100006, ""customer_id"": 38, ""city"": ""Chennai"", ""product_id"": 4, ""quantity"": 2, ""order_amount"": 2998, ""payment_method"": null}",835,atliq.orders.events,0,2026-08-27T01:04:41.187Z
100007,"{""event_id"": ""f02d3a14-43b1-4c66-8226-9520b4a92a68"", ""event_type"": ""order_shipped"", ""event_ts"": ""2026-08-27T01:04:42.197Z"", ""order_id"": 100007, ""customer_id"": 14, ""city"": ""Hyderabad"", ""product_id"": 6, ""quantity"": 1, ""order_amount"": 899, ""payment_method"": null}",836,atliq.orders.events,0,2026-08-27T01:04:42.197Z
100005,"{""event_id"": ""e1bfe6ab-eef5-44e2-afda-20dea7cdb67e"", ""event_type"": ""order_shipped"", ""event_ts"": ""2026-08-27T01:04:45.732Z"", ""order_id"": 100005, ""customer_id"": 28, ""city"": ""Jaipur"", ""product_id"": 14, ""quantity"": 1, ""order_amount"": 1599, ""payment_method"": null}",837,atliq.orders.events,0,2026-08-27T01:04:45.733Z


## TASK 2 — Silver: parse, de-duplicate, handle late data
Stream FROM the Bronze table into `atliq.streaming.silver_order_events`:
1. Parse the JSON value with an explicit schema (event_id, event_type,
   event_ts, order_id, customer_id, city, product_id, quantity,
   order_amount, payment_method).
2. Convert event_ts to a real timestamp.
3. Add a **10-minute watermark** on event_ts, then
   **dropDuplicates(["event_id"])** — so a replayed event can never land twice.

Hint: `spark.readStream.table(...)`, `F.from_json`, `withWatermark`.

In [0]:

bronze_order_events_df = spark.readStream.table(f"{CATALOG}.{SCHEMA}.bronze_order_events")

order_schema = (
                    StructType()
                    .add("event_id", StringType(), True)
                    .add("event_type", StringType(), True)
                    .add("event_ts", StringType(), True)
                    .add("order_id", LongType(), True)
                    .add("customer_id", IntegerType(), True)
                    .add("city", StringType(), True)
                    .add("product_id", IntegerType(), True)
                    .add("quantity", IntegerType(), True)
                    .add("order_amount", DecimalType(10,2), True)
                    .add("payment_method", StringType(), True)
                )


silver_df = (
                bronze_order_events_df
                    .select(
                            F.from_json(F.col("json_value"), order_schema).alias("data"), 
                            F.col("kafka_key"), 
                            F.col("offset"), 
                            F.col("topic"), 
                            F.col("partition"), 
                            F.col("timestamp"))
                    .select("data.*", "kafka_key", "offset", "topic", "partition", "timestamp")
                    .withColumn("event_ts", F.to_timestamp("event_ts"))
                    .withWatermark("event_ts", "10 minutes")
                    .dropDuplicates(['event_id'])
            )

query = (
            (
                silver_df.writeStream
                        .format('delta')
                        .outputMode('append')
                        .trigger(availableNow=True)
                        .option("checkpointLocation", f"{CKPT}/silver")
                        .toTable(f"{CATALOG}.{SCHEMA}.silver_order_events")
            )
        )

query.awaitTermination()

silver_order_events_count = spark.table(f"{CATALOG}.{SCHEMA}.silver_order_events").count()

print(f"Silver order events count: {silver_order_events_count}")

display(
    spark.table(f"{CATALOG}.{SCHEMA}.silver_order_events")
    .limit(10)
)


Silver order events count: 5321


event_id,event_type,event_ts,order_id,customer_id,city,product_id,quantity,order_amount,payment_method,kafka_key,offset,topic,partition,timestamp
9e07d1ca-bbcd-4cf1-a990-ed5f9e0d5b38,order_placed,2026-08-26T04:35:34.914Z,100001,36,Kolkata,7,3,3897.00,null,100001,0,atliq.orders.events,5,2026-08-26T04:35:34.914Z
475a9bc8-f853-4833-acef-0ffd001e3448,payment_received,2026-08-26T04:36:00.630Z,100031,36,Bengaluru,1,3,7497.00,UPI,100031,8,atliq.orders.events,2,2026-08-26T04:36:00.630Z
45234613-758e-4355-a6a4-d45d4c609670,order_placed,2026-08-26T04:35:37.942Z,100005,7,Surat,24,1,599.00,null,100005,0,atliq.orders.events,0,2026-08-26T04:35:37.942Z
e052eaec-f3ce-401e-91e1-c3e2ab41365d,payment_received,2026-08-26T04:36:01.135Z,100032,40,Delhi,24,3,1797.00,Net Banking,100032,10,atliq.orders.events,2,2026-08-26T04:36:01.135Z
ff7990d0-5d03-4056-83ab-0c0e91ec55da,order_cancelled,2026-08-26T04:36:17.258Z,100046,39,Ahmedabad,21,1,249.00,null,100046,27,atliq.orders.events,0,2026-08-26T04:36:17.258Z
93fe315a-457a-4154-a637-4d1c3dd6375e,order_placed,2026-08-26T04:36:18.268Z,100056,18,Delhi,20,3,5697.00,null,100056,17,atliq.orders.events,5,2026-08-26T04:36:18.268Z
102423cd-fc21-4331-bf7b-cd96118f0c81,payment_received,2026-08-26T04:35:56.603Z,100027,36,Chennai,24,2,1198.00,UPI,100027,16,atliq.orders.events,0,2026-08-26T04:35:56.603Z
01d486d0-8a5e-4bd6-b7a9-3d2f5dbe54cc,order_placed,2026-08-26T04:35:40.963Z,100010,10,Surat,5,2,9998.00,null,100010,2,atliq.orders.events,4,2026-08-26T04:35:40.964Z
d2db6503-869c-45da-b3b6-f2333812258f,order_placed,2026-08-26T04:36:10.713Z,100044,29,Bengaluru,16,1,299.00,null,100044,14,atliq.orders.events,4,2026-08-26T04:36:10.713Z
c2d23066-fd1d-492a-879f-1022ac3e4217,payment_received,2026-08-26T04:36:12.726Z,100048,14,Pune,2,3,9897.00,UPI,100048,25,atliq.orders.events,3,2026-08-26T04:36:12.726Z


## TASK 3 — Gold: the live revenue ticker
From the Silver stream, keep only `payment_received` events and aggregate
into **5-minute tumbling windows**: orders_paid = count, revenue = sum of
order_amount. Append closed windows to `atliq.streaming.gold_revenue_5min`.

Hint: `F.window("event_ts", "5 minutes")` — and think about WHY a window
only appears after the watermark passes its end (you will explain this
in your write-up).

In [0]:

silver_order_events_df = spark.readStream.table(f"{CATALOG}.{SCHEMA}.silver_order_events")

gold_df =  (
            silver_order_events_df
                .filter(F.col("event_type") == "payment_received")
                .withWatermark("event_ts", "10 minutes")
                .groupBy(
                    F.window(
                        F.col("event_ts"),
                        "5 minutes"
                    )
                )
                .agg(
                    F.count("*").alias("orders_paid"),
                    F.sum("order_amount").alias("revenue")
                )
                .select(
                    F.col("window.start").alias("window_start"),
                    F.col("window.end").alias("window_end"),
                    "orders_paid",
                    "revenue"
                )
            )

query = (
            (
                gold_df.writeStream
                        .format('delta')
                        .outputMode('append')
                        .trigger(availableNow=True)
                        .option("checkpointLocation", f"{CKPT}/gold")
                        .toTable(f"{CATALOG}.{SCHEMA}.gold_revenue_5min")
            )
        )

query.awaitTermination()


gold_revenue_5min_count = spark.table(f"{CATALOG}.{SCHEMA}.gold_revenue_5min").count()

print(f"Gold order events count: {gold_revenue_5min_count}")

display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_revenue_5min")
    .limit(10)
)



Gold order events count: 7


window_start,window_end,orders_paid,revenue
2026-08-26T04:40:00.000Z,2026-08-26T04:45:00.000Z,283,688818.00
2026-08-26T04:35:00.000Z,2026-08-26T04:40:00.000Z,268,618274.00
2026-08-26T04:50:00.000Z,2026-08-26T04:55:00.000Z,283,715551.00
2026-08-26T04:45:00.000Z,2026-08-26T04:50:00.000Z,260,665466.00
2026-08-26T05:00:00.000Z,2026-08-26T05:05:00.000Z,284,729962.00
2026-08-26T04:55:00.000Z,2026-08-26T05:00:00.000Z,258,673343.00
2026-08-26T05:05:00.000Z,2026-08-26T05:10:00.000Z,28,62788.00


## Verify (given)

In [0]:
%sql
SELECT event_type, COUNT(*) AS events
FROM atliq.streaming.silver_order_events GROUP BY event_type;

event_type,events
order_placed,1999
payment_received,1690
order_cancelled,351
order_shipped,1281


In [0]:
%sql
SELECT * FROM atliq.streaming.gold_revenue_5min ORDER BY window_start DESC LIMIT 12;

window_start,window_end,orders_paid,revenue
2026-08-26T05:05:00.000Z,2026-08-26T05:10:00.000Z,28,62788.00
2026-08-26T05:00:00.000Z,2026-08-26T05:05:00.000Z,284,729962.00
2026-08-26T04:55:00.000Z,2026-08-26T05:00:00.000Z,258,673343.00
2026-08-26T04:50:00.000Z,2026-08-26T04:55:00.000Z,283,715551.00
2026-08-26T04:45:00.000Z,2026-08-26T04:50:00.000Z,260,665466.00
2026-08-26T04:40:00.000Z,2026-08-26T04:45:00.000Z,283,688818.00
2026-08-26T04:35:00.000Z,2026-08-26T04:40:00.000Z,268,618274.00


In [0]:
%sql
SELECT * FROM atliq.streaming.gold_daily_summary LIMIT 10;

event_date,orders_placed,orders_paid,orders_cancelled,revenue
2026-08-27,33,26,7,86247.00
2026-08-26,1966,1664,344,4154202.00
